Random Forest

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Load dataset
df = pd.read_csv("synthetic_depression_dataset.csv")

X = df.drop("label", axis=1)
y = df["label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train model
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.91      0.78        79
           1       0.53      0.20      0.29        41

    accuracy                           0.67       120
   macro avg       0.61      0.55      0.53       120
weighted avg       0.63      0.67      0.61       120

ROC-AUC Score: 0.7130287125656067


XGBoost

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("synthetic_depression_dataset.csv")

X = df.drop("label", axis=1)
y = df["label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train model
model = XGBClassifier(
    scale_pos_weight= (len(y_train[y_train==0]) / len(y_train[y_train==1])),
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.75      0.74        79
           1       0.50      0.49      0.49        41

    accuracy                           0.66       120
   macro avg       0.62      0.62      0.62       120
weighted avg       0.66      0.66      0.66       120

ROC-AUC Score: 0.6875578882371102


SGBoost with 5 folds

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("synthetic_depression_dataset.csv")

X = df.drop("label", axis=1).values
y = df["label"].values

# Stratified K-Fold setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []
roc_aucs = []

fold = 1

for train_index, test_index in skf.split(X, y):

    print(f"\n========== Fold {fold} ==========")

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Handle imbalance
    scale_pos_weight = (len(y_train[y_train==0]) / len(y_train[y_train==1]))

    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(classification_report(y_test, y_pred))

    acc = np.mean(y_pred == y_test)
    roc = roc_auc_score(y_test, y_prob)

    print("Fold Accuracy:", acc)
    print("Fold ROC-AUC:", roc)

    accuracies.append(acc)
    roc_aucs.append(roc)

    fold += 1

print("\n==============================")
print("Average Accuracy:", np.mean(accuracies))
print("Average ROC-AUC:", np.mean(roc_aucs))
print("==============================")


========== Fold 1 ==========
              precision    recall  f1-score   support

           0       0.81      0.86      0.84        80
           1       0.69      0.60      0.64        40

    accuracy                           0.78       120
   macro avg       0.75      0.73      0.74       120
weighted avg       0.77      0.78      0.77       120

Fold Accuracy: 0.775
Fold ROC-AUC: 0.8059375

========== Fold 2 ==========
              precision    recall  f1-score   support

           0       0.75      0.79      0.77        80
           1       0.53      0.47      0.50        40

    accuracy                           0.68       120
   macro avg       0.64      0.63      0.63       120
weighted avg       0.68      0.68      0.68       120

Fold Accuracy: 0.6833333333333333
Fold ROC-AUC: 0.75125

========== Fold 3 ==========
              precision    recall  f1-score   support

           0       0.77      0.82      0.80        79
           1       0.61      0.54      0.57   